In [ ]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 86.6 MB/s eta 0:00:00


In [ ]:
# SpaceX Launch Records Dashboard
# Built with Plotly Dash

import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# ── Load data directly from URL ───────────────────────────────────────────────
DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"

spacex_df = pd.read_csv(DATA_URL)

max_payload = spacex_df["Payload Mass (kg)"].max()
min_payload = spacex_df["Payload Mass (kg)"].min()

# ── App layout ────────────────────────────────────────────────────────────────
app = dash.Dash(__name__)

app.layout = html.Div(children=[

    html.H1(
        "SpaceX Launch Records Dashboard",
        style={
            "textAlign": "center",
            "color": "#503D36",
            "fontSize": 40
        }
    ),

    # Dropdown for launch sites
    dcc.Dropdown(
        id="site-dropdown",
        options=[{"label": "All Sites", "value": "ALL"}] + [
            {"label": site, "value": site}
            for site in sorted(spacex_df["Launch Site"].unique())
        ],
        value="ALL",
        placeholder="Select a Launch Site here",
        searchable=True
    ),

    html.Br(),

    # Pie chart
    html.Div(dcc.Graph(id="success-pie-chart")),

    html.Br(),

    # Payload slider
    html.P("Payload range (Kg):"),

    dcc.RangeSlider(
        id="payload-slider",
        min=min_payload,
        max=max_payload,
        step=1000,
        marks={
            int(i): str(int(i))
            for i in range(
                int(min_payload),
                int(max_payload) + 1,
                2500
            )
        },
        value=[min_payload, max_payload]
    ),

    html.Br(),

    # Scatter chart
    html.Div(dcc.Graph(id="success-payload-scatter-chart")),

])

# ── Pie Chart Callback ────────────────────────────────────────────────────────
@app.callback(
    Output("success-pie-chart", "figure"),
    Input("site-dropdown", "value")
)
def get_pie_chart(entered_site):

    if entered_site == "ALL":

        fig = px.pie(
            spacex_df[spacex_df["class"] == 1],
            names="Launch Site",
            title="Total Successful Launches by Site"
        )

    else:

        filtered_df = spacex_df[
            spacex_df["Launch Site"] == entered_site
        ]

        outcome_counts = (
            filtered_df["class"]
            .value_counts()
            .reset_index()
        )

        outcome_counts.columns = ["class", "count"]

        outcome_counts["Outcome"] = outcome_counts["class"].map({
            1: "Success",
            0: "Failure"
        })

        fig = px.pie(
            outcome_counts,
            values="count",
            names="Outcome",
            title=f"Launch Outcomes for {entered_site}"
        )

    return fig


# ── Scatter Plot Callback ─────────────────────────────────────────────────────
@app.callback(
    Output("success-payload-scatter-chart", "figure"),
    [
        Input("site-dropdown", "value"),
        Input("payload-slider", "value")
    ]
)
def get_scatter_chart(entered_site, payload_range):

    low, high = payload_range

    filtered_df = spacex_df[
        (spacex_df["Payload Mass (kg)"] >= low) &
        (spacex_df["Payload Mass (kg)"] <= high)
    ]

    if entered_site == "ALL":

        fig = px.scatter(
            filtered_df,
            x="Payload Mass (kg)",
            y="class",
            color="Booster Version Category",
            title="Payload vs Launch Outcome for All Sites"
        )

    else:

        site_df = filtered_df[
            filtered_df["Launch Site"] == entered_site
        ]

        fig = px.scatter(
            site_df,
            x="Payload Mass (kg)",
            y="class",
            color="Booster Version Category",
            title=f"Payload vs Launch Outcome for {entered_site}"
        )

    return fig


# ── Run App ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app.run(debug=True, port=8050)